In [1]:
# Task 11: Execution Plan Analysis
# For at least 3 major queries: Use: df.explain("extended")
# Identify:
# - Exchange
# - BroadcastHashJoin
# - SortMergeJoin
# - WholeStageCodegen
# Explain each component.



In [11]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum, avg
from pyspark.sql.window import Window

In [12]:
spark = SparkSession.builder \
    .appName("ExecutionPlan") \
    .master("yarn") \
    .getOrCreate()
spark

26/02/24 11:56:57 WARN Client: Neither spark.yarn.jars nor spark.yarn.archive is set, falling back to uploading libraries under SPARK_HOME.


In [13]:
df = spark.read.parquet("hdfs:///data/covid/staging/full_grouped")


### For at least 3 major queries: Use: df.explain("extended")

In [14]:
# 1. Aggregation Query
death_df = df.groupBy("Country/Region") \
             .agg(sum("Deaths").alias("total_deaths"))

death_df.explain("extended")

# Execution Plan Components to Identify:
# - Exchange:
#   Indicates shuffle due to groupBy (wide transformation).
#   Data is hash-partitioned by Country/Region.
#
# - HashAggregate:
#   Performs partial + final aggregation.
#
# - WholeStageCodegen:
#   Spark combines multiple physical operators into a single JVM function
#   for CPU efficiency.


== Parsed Logical Plan ==
'Aggregate ['Country/Region], ['Country/Region, 'sum('Deaths) AS total_deaths#59]
+- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New deaths#56L,New recovered#57L,WHO Region#58] parquet

== Analyzed Logical Plan ==
Country/Region: string, total_deaths: bigint
Aggregate [Country/Region#50], [Country/Region#50, sum(Deaths#52L) AS total_deaths#59L]
+- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New deaths#56L,New recovered#57L,WHO Region#58] parquet

== Optimized Logical Plan ==
Aggregate [Country/Region#50], [Country/Region#50, sum(Deaths#52L) AS total_deaths#59L]
+- Project [Country/Region#50, Deaths#52L]
   +- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New deaths#56L,New recovered#57L,WHO Region#58] parquet

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[Country/R

In [15]:
# 2. Join Query
world_df = spark.read.parquet("hdfs:///data/covid/staging/worldometer_data")

joined_df = df.join(
    world_df,
    df["Country/Region"] == world_df["Country/Region"],
    "inner"
)

joined_df.explain("extended")

# Execution Plan Components to Identify:
#
# - Exchange:
#   Shuffle on both sides if Spark chooses SortMergeJoin.
#
# - SortMergeJoin:
#   Used when both datasets are large.
#   Requires shuffle + sort before merge.
#
# - BroadcastHashJoin (if applicable):
#   Appears when smaller dataset is auto-broadcasted.
#   Avoids shuffle on small side.
#
# - WholeStageCodegen:
#   Optimizes execution by generating compiled JVM bytecode.

[Stage 1:>                                                          (0 + 1) / 1]

== Parsed Logical Plan ==
Join Inner, (Country/Region#50 = Country/Region#74)
:- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New deaths#56L,New recovered#57L,WHO Region#58] parquet
+- Relation [Country/Region#74,Continent#75,Population#76L,TotalCases#77L,NewCases#78L,TotalDeaths#79L,NewDeaths#80L,TotalRecovered#81L,NewRecovered#82L,ActiveCases#83L] parquet

== Analyzed Logical Plan ==
Date: date, Country/Region: string, Confirmed: bigint, Deaths: bigint, Recovered: bigint, Active: bigint, New cases: bigint, New deaths: bigint, New recovered: bigint, WHO Region: string, Country/Region: string, Continent: string, Population: bigint, TotalCases: bigint, NewCases: bigint, TotalDeaths: bigint, NewDeaths: bigint, TotalRecovered: bigint, NewRecovered: bigint, ActiveCases: bigint
Join Inner, (Country/Region#50 = Country/Region#74)
:- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New de

In [16]:
# 3. Window Function Query
windowSpec = Window.partitionBy("Country/Region") \
                   .orderBy("Date") \
                   .rowsBetween(-6, 0)

rolling_df = df.withColumn(
    "rolling_avg_recovery",
    avg("Recovered").over(windowSpec)
)

rolling_df.explain("extended")

# Execution Plan Components to Identify:
#
# - Exchange:
#   Occurs if partitioning requires redistribution by Country/Region.
#
# - Sort:
#   Required due to orderBy(Date) inside window.
#
# - Window:
#   Applies rolling aggregation within partition.
#
# - WholeStageCodegen:
#   Combines sort + window operators into optimized execution pipeline.

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(rolling_avg_recovery, 'avg('Recovered) windowspecdefinition('Country/Region, 'Date ASC NULLS FIRST, specifiedwindowframe(RowFrame, -6, currentrow$())), None)]
+- Relation [Date#49,Country/Region#50,Confirmed#51L,Deaths#52L,Recovered#53L,Active#54L,New cases#55L,New deaths#56L,New recovered#57L,WHO Region#58] parquet

== Analyzed Logical Plan ==
Date: date, Country/Region: string, Confirmed: bigint, Deaths: bigint, Recovered: bigint, Active: bigint, New cases: bigint, New deaths: bigint, New recovered: bigint, WHO Region: string, rolling_avg_recovery: double
Project [Date#49, Country/Region#50, Confirmed#51L, Deaths#52L, Recovered#53L, Active#54L, New cases#55L, New deaths#56L, New recovered#57L, WHO Region#58, rolling_avg_recovery#85]
+- Project [Date#49, Country/Region#50, Confirmed#51L, Deaths#52L, Recovered#53L, Active#54L, New cases#55L, New deaths#56L, New recovered#57L, WHO Region#58, rolling_avg_recovery#85, rolling_a

In [17]:
spark.stop()